# 📓 Notebook 1 — Chuẩn bị dữ liệu (tự crawl)

**Đề tài**: Phân loại rau sạch / rau hỏng bằng Deep Learning

Notebook này thực hiện end-to-end khâu chuẩn bị, **TỰ CRAWL** ảnh từ web (theo yêu cầu đề bài):

1. Setup môi trường + cài deps
2. **Crawl ảnh** từ Google + Bing + Baidu Images theo nhiều keyword
3. Sanity check: thống kê theo keyword, kích thước, file lỗi
4. Tiền xử lý: resize 224×224, loại trùng/lỗi, split train/valid/test 70/15/15
5. Trực quan hoá: ảnh mẫu, phân bố class, augmentation demo

Sau khi chạy xong → tiếp tục với **`02_train_and_evaluate.ipynb`**.

💡 Crawl 10.000 ảnh mất ~30-60 phút. Có thể chạy `--target` nhỏ hơn để test nhanh.

## 1. Thiết lập đường dẫn project

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path('..').resolve()
if Path.cwd().name == 'notebook':
    os.chdir(ROOT)
sys.path.insert(0, str(Path.cwd()))

print('📁 CWD :', Path.cwd())
print('🐍 Py  :', sys.version.split()[0])

## 2. Cài đặt dependencies (lần đầu)

In [ ]:
# Bỏ comment dòng dưới nếu chưa cài
# !pip install -q -r requirements.txt

import tensorflow as tf
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image

print('TF      :', tf.__version__)
print('GPU     :', tf.config.list_physical_devices('GPU') or '(CPU only)')

## 3. Crawl ảnh từ Google + Bing + Baidu

Cấu hình trong `crawler/crawl_images.py`:
- **41 keyword** đa dạng (Anh + Việt) cho fresh và rotten
- 3 search engine song song để tăng đa dạng
- Mỗi keyword lưu thành `<class>/<slug>_<idx>.jpg` để sau biết ảnh thuộc keyword nào

💡 `--target 10000` → mỗi class ~5000 ảnh. Có thể giảm xuống `2000` để test trước.

In [ ]:
# Bỏ comment để chạy crawl. Mất 30-60 phút lần đầu.
# !python crawler/crawl_images.py --target 10000 --engines google,bing,baidu

# Hoặc test nhanh:
# !python crawler/crawl_images.py --target 2000 --engines google,bing

## 4. Kiểm tra dataset đã crawl

In [ ]:
from collections import Counter
RAW = Path('dataset/raw')
assert RAW.exists(), 'Chưa có dataset/raw — chạy cell crawl trước'

stats = {d.name: sum(1 for f in d.iterdir() if f.is_file())
         for d in RAW.iterdir() if d.is_dir()}
print('📦 Số ảnh từng class:')
for k, v in stats.items():
    print(f'   {k:<8s}: {v:,}')
print(f'   {"TỔNG":<8s}: {sum(stats.values()):,}')

In [ ]:
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
import re

broken, small, sizes, keywords = [], [], [], Counter()
for cls_dir in sorted(d for d in RAW.iterdir() if d.is_dir()):
    for f in tqdm(list(cls_dir.iterdir()), desc=cls_dir.name):
        try:
            with Image.open(f) as im:
                w, h = im.size
                sizes.append((w, h))
                if min(w, h) < 64: small.append(f)
        except (UnidentifiedImageError, OSError):
            broken.append(f)
        m = re.match(r'(.+)_\d+$', f.stem)
        kw = m.group(1) if m else f.stem
        keywords[kw] += 1

print(f'\n✅ Tổng ảnh quét  : {len(sizes):,}')
print(f'❌ Ảnh hỏng       : {len(broken)}')
print(f'⚠️  Ảnh < 64px    : {len(small)}')
print(f'🔤 Số keyword     : {len(keywords)}')
print(f'\n🔤 Top 15 keyword nhiều ảnh nhất:')
for kw, n in keywords.most_common(15):
    print(f'   {kw:<35s}: {n:,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bars = axes[0].bar(stats.keys(), stats.values(), color=['#2E7D32', '#C62828'])
for b, v in zip(bars, stats.values()):
    axes[0].text(b.get_x()+b.get_width()/2, v, f'{v:,}', ha='center', va='bottom')
axes[0].set_title('Phân bố số ảnh theo class', fontweight='bold')
axes[0].set_ylabel('Số ảnh')
ws = [w for w, h in sizes]
axes[1].hist(ws, bins=40, color='#1976D2', edgecolor='white')
axes[1].set_title('Phân bố chiều rộng ảnh (px)', fontweight='bold')
axes[1].set_xlabel('Chiều rộng (px)'); axes[1].set_ylabel('Số ảnh')
axes[1].axvline(224, color='red', ls='--', label='Target 224')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
import random
random.seed(42)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for row, cls in enumerate(['fresh', 'rotten']):
    files = random.sample([f for f in (RAW / cls).iterdir() if f.is_file()], 8)
    for col, f in enumerate(files):
        try: axes[row, col].imshow(Image.open(f))
        except Exception: pass
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls.upper(), loc='left', fontweight='bold',
                                     fontsize=12, color='#2E7D32' if cls=='fresh' else '#C62828')
plt.suptitle('Ảnh mẫu mỗi class (sau crawl)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4.5. EDA — Phân tích khám phá dữ liệu

Mục đích: hiểu sâu dataset trước khi train. Sinh các biểu đồ trả lời 7 câu hỏi:

1. Mỗi class có bao nhiêu ảnh? (đã xem ở 4.1)
2. Phân bố nguồn (crawl tự thu thập / augmentation / web extra)?
3. Kích thước ảnh phân bố ra sao?
4. Aspect ratio (W/H) phân bố thế nào?
5. Format file đa dạng cỡ nào?
6. Màu sắc trung bình fresh vs rotten có khác?
7. Top keyword nhiều ảnh nhất?

Toàn bộ output lưu vào `results/eda_*.png` để chèn vào báo cáo.


In [ ]:
# Setup EDA
import os, json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
RAW = Path('dataset/raw')
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 120,
                     'axes.spines.top': False, 'axes.spines.right': False})

CLASSES = ['fresh', 'rotten']
CLASS_COLOR = {'fresh': '#2E7D32', 'rotten': '#C62828'}

# Quét toàn bộ metadata 1 lần để tái sử dụng
print('Đang quét toàn bộ dataset/raw để build inventory...')
records = []
for cls in CLASSES:
    cls_dir = RAW / cls
    for f in tqdm(list(cls_dir.iterdir()), desc=cls):
        if not f.is_file():
            continue
        # Phân loại nguồn theo prefix tên file
        stem = f.stem
        if stem.startswith('aug_'):       source = 'augmentation'
        elif stem.startswith('web_extra_'): source = 'web_extra'
        else:                              source = 'crawl'
        # Mở ảnh để lấy kích thước + format
        try:
            with Image.open(f) as im:
                w, h = im.size
                fmt = (im.format or 'UNKNOWN').upper()
        except (UnidentifiedImageError, OSError):
            continue
        records.append({'class': cls, 'file': f.name, 'source': source,
                        'width': w, 'height': h, 'format': fmt,
                        'aspect': round(w / h, 3),
                        'size_kb': f.stat().st_size // 1024})
df = pd.DataFrame(records)
print(f'Tổng: {len(df):,} ảnh hợp lệ')
df.to_csv(RESULTS / 'eda_inventory.csv', index=False, encoding='utf-8')
print(f'-> Đã lưu inventory: {RESULTS / "eda_inventory.csv"}')
df.head()


In [ ]:
# EDA 1: Phân bố nguồn (crawl / augmentation / web_extra)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Subplot 1: Stacked bar — nguồn theo class
pivot = df.groupby(['class', 'source']).size().unstack(fill_value=0)
pivot = pivot[['crawl', 'augmentation', 'web_extra']]  # thứ tự ổn định
pivot.plot(kind='bar', stacked=True, ax=axes[0],
           color=['#1976D2', '#FFA726', '#9E9E9E'], edgecolor='white')
axes[0].set_title('Nguồn ảnh theo class', fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Số ảnh')
axes[0].set_xticklabels(pivot.index, rotation=0)
for c in axes[0].containers:
    axes[0].bar_label(c, label_type='center', fontsize=9, color='white',
                      fontweight='bold')

# Subplot 2: Pie tổng
src_total = df['source'].value_counts()
colors = {'crawl': '#1976D2', 'augmentation': '#FFA726', 'web_extra': '#9E9E9E'}
axes[1].pie(src_total, labels=src_total.index,
            colors=[colors[s] for s in src_total.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title(f'Tỉ lệ nguồn (tổng {len(df):,} ảnh)', fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS / 'eda_source_distribution.png', bbox_inches='tight')
plt.show()
print(f'-> Đã lưu: {RESULTS / "eda_source_distribution.png"}')
print('\nChi tiết:')
print(pivot)


In [ ]:
# EDA 2: Phân bố kích thước (width, height, aspect ratio)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, col, title, target in zip(
    axes, ['width', 'height', 'aspect'],
    ['Chiều rộng (px)', 'Chiều cao (px)', 'Aspect ratio (W/H)'],
    [224, 224, 1.0]
):
    for cls in CLASSES:
        ax.hist(df[df['class'] == cls][col], bins=40, alpha=0.6,
                label=cls, color=CLASS_COLOR[cls], edgecolor='white')
    ax.axvline(target, color='red', ls='--', alpha=0.7,
               label=f'Target {target}')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(col); ax.set_ylabel('Số ảnh')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS / 'eda_size_distribution.png', bbox_inches='tight')
plt.show()
print(f'-> Đã lưu: {RESULTS / "eda_size_distribution.png"}')

# In thống kê mô tả
print('\nThống kê kích thước:')
print(df[['width', 'height', 'aspect', 'size_kb']].describe().round(2))


In [ ]:
# EDA 3: Phân bố format file
fmt_count = df.groupby(['class', 'format']).size().unstack(fill_value=0)
print('Format theo class:')
print(fmt_count)

fig, ax = plt.subplots(figsize=(8, 4.5))
fmt_count.plot(kind='bar', ax=ax, edgecolor='white',
               color=['#1976D2', '#7B1FA2', '#388E3C', '#F57C00', '#5D4037'])
ax.set_title('Phân bố định dạng file theo class', fontweight='bold')
ax.set_xlabel('Class'); ax.set_ylabel('Số ảnh')
ax.set_xticklabels(fmt_count.index, rotation=0)
ax.legend(title='Format', loc='best')
for c in ax.containers:
    ax.bar_label(c, label_type='edge', fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS / 'eda_format_distribution.png', bbox_inches='tight')
plt.show()
print(f'\n-> Đã lưu: {RESULTS / "eda_format_distribution.png"}')
print('\nNhận xét: ảnh sẽ được chuẩn hoá về JPEG khi resize.')


In [ ]:
# EDA 4: Color stats — RGB mean fresh vs rotten
# Sample 500 ảnh mỗi class để tính (đủ chính xác, nhanh)
import random
random.seed(42)

color_stats = []
for cls in CLASSES:
    files = list((RAW / cls).iterdir())
    sample = random.sample(files, min(500, len(files)))
    for f in tqdm(sample, desc=f'color {cls}'):
        try:
            with Image.open(f) as im:
                im = im.convert('RGB').resize((64, 64))  # nhỏ cho nhanh
                arr = np.asarray(im)
                color_stats.append({
                    'class': cls,
                    'R_mean': arr[..., 0].mean(),
                    'G_mean': arr[..., 1].mean(),
                    'B_mean': arr[..., 2].mean(),
                    'brightness': arr.mean(),
                })
        except (UnidentifiedImageError, OSError):
            continue

df_color = pd.DataFrame(color_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Subplot 1: Box plot RGB channels
df_long = df_color.melt(id_vars='class',
                        value_vars=['R_mean', 'G_mean', 'B_mean'],
                        var_name='channel', value_name='mean_value')
positions, labels, colors = [], [], []
data, box_colors = [], []
i = 0
for ch in ['R_mean', 'G_mean', 'B_mean']:
    for cls in CLASSES:
        data.append(df_color[df_color['class'] == cls][ch].values)
        positions.append(i); i += 1
        labels.append(f'{ch[0]}\n{cls}')
        box_colors.append(CLASS_COLOR[cls])
    i += 0.5  # khoảng cách giữa các channel
bp = axes[0].boxplot(data, positions=positions, patch_artist=True, widths=0.7)
for patch, c in zip(bp['boxes'], box_colors):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0].set_xticks(positions); axes[0].set_xticklabels(labels, fontsize=9)
axes[0].set_title('RGB mean theo class', fontweight='bold')
axes[0].set_ylabel('Giá trị pixel (0–255)')

# Subplot 2: Brightness comparison
for cls in CLASSES:
    axes[1].hist(df_color[df_color['class'] == cls]['brightness'],
                 bins=30, alpha=0.6, label=cls,
                 color=CLASS_COLOR[cls], edgecolor='white')
axes[1].set_title('Phân bố độ sáng (brightness mean)', fontweight='bold')
axes[1].set_xlabel('Brightness (0–255)'); axes[1].set_ylabel('Số ảnh')
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS / 'eda_color_stats.png', bbox_inches='tight')
plt.show()
print(f'-> Đã lưu: {RESULTS / "eda_color_stats.png"}')

# Insight
print('\nMean RGB per class:')
print(df_color.groupby('class')[['R_mean', 'G_mean', 'B_mean', 'brightness']]
      .mean().round(1))


In [ ]:
# EDA 5: Top keyword + summary
import re
df['keyword'] = df['file'].apply(
    lambda x: re.sub(r'_\d+(\.[a-z]+)?$', '', Path(x).stem)
)
# Loại các prefix riêng để gom nhóm tốt hơn
df['keyword_clean'] = (df['keyword']
    .str.replace(r'^aug_', '', regex=True)
    .str.replace(r'^web_extra_\d*', 'web_extra', regex=True))

top_kw = df.groupby('keyword_clean').size().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
top_kw.plot(kind='barh', ax=ax, color='#1976D2', edgecolor='white')
ax.set_title('Top 20 keyword nhiều ảnh nhất', fontweight='bold')
ax.set_xlabel('Số ảnh'); ax.invert_yaxis()
for c in ax.containers:
    ax.bar_label(c, label_type='edge', fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS / 'eda_top_keywords.png', bbox_inches='tight')
plt.show()
print(f'-> Đã lưu: {RESULTS / "eda_top_keywords.png"}')

# Save summary JSON cho báo cáo
summary = {
    'total_images': int(len(df)),
    'by_class': df.groupby('class').size().to_dict(),
    'by_source': df.groupby('source').size().to_dict(),
    'by_format': df.groupby('format').size().to_dict(),
    'size_stats': {
        'width_mean': float(df['width'].mean()),
        'height_mean': float(df['height'].mean()),
        'aspect_mean': float(df['aspect'].mean()),
        'aspect_median': float(df['aspect'].median()),
    },
    'unique_keywords': int(df['keyword_clean'].nunique()),
}
(RESULTS / 'eda_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'-> Đã lưu summary: {RESULTS / "eda_summary.json"}')
print('\nTỔNG KẾT EDA:')
for k, v in summary.items():
    print(f'  {k}: {v}')


## 5. Tiền xử lý + split train/valid/test

In [ ]:
!python preprocessing/preprocess.py --src dataset/raw --dst dataset --img-size 224

In [ ]:
split_stats = {}
for split in ('train', 'valid', 'test'):
    sd = Path(f'dataset/{split}')
    if not sd.exists(): continue
    split_stats[split] = {c.name: sum(1 for _ in c.iterdir())
                          for c in sd.iterdir() if c.is_dir()}
df_split = pd.DataFrame(split_stats).fillna(0).astype(int)
print('📊 Phân bố sau split:'); print(df_split)
print(f'\nTỔNG: {df_split.values.sum():,} ảnh')
ax = df_split.plot.bar(figsize=(8, 4),
                       color={'train': '#1976D2', 'valid': '#FFA726', 'test': '#7B1FA2'},
                       edgecolor='white')
ax.set_title('Phân bố train / valid / test', fontweight='bold')
ax.set_ylabel('Số ảnh'); ax.set_xlabel('Class')
ax.set_xticklabels(df_split.index, rotation=0)
for c in ax.containers:
    ax.bar_label(c, label_type='edge', fontsize=8)
plt.tight_layout(); plt.show()

## 6. Demo Augmentation

In [ ]:
from preprocessing.augmentation import build_train_generator
gen = build_train_generator(Path('dataset/train'), img_size=224, batch_size=1)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for ax in axes.ravel():
    x, _ = next(gen)
    ax.imshow(x[0]); ax.axis('off')
plt.suptitle('8 phiên bản sau augmentation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## ✅ Hoàn thành chuẩn bị

- `dataset/raw/{fresh,rotten}/` — ảnh gốc tự crawl từ web
- `dataset/processed/` — ảnh đã clean + resize 224×224
- `dataset/{train,valid,test}/` — đã split 70/15/15

👉 **Tiếp theo**: mở `02_train_and_evaluate.ipynb`.